# Batch run + all-location summary (pipeline v2)

Use `run_rider_count_new.ipynb` for a single location; this notebook covers: **batch-run all non-excluded locations -> all-location summary table -> comparison against the manual counts -> review-CSV metrics**.

Supports resume: completed locations are skipped automatically (set FORCE_RERUN=True to recompute).

In [ ]:
# Cell 1: Bootstrap
import sys, importlib.util
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))
script = REPO_ROOT / "scripts" / "run_rider_count_new.py"
if not script.exists():
    hits = list(REPO_ROOT.rglob("run_rider_count_new.py"))
    assert hits, "run_rider_count_new.py not found"
    script = hits[0]
spec = importlib.util.spec_from_file_location("run_rider_count_new", script)
rrc = importlib.util.module_from_spec(spec)
sys.modules["run_rider_count_new"] = rrc
spec.loader.exec_module(rrc)
print("Repo root:", REPO_ROOT, "| script:", script)

In [ ]:
# Cell 2: Configuration
from types import SimpleNamespace

DATA_ROOT = Path(r"D:\0_MAIN_BIKE_DATASETS_clean")
OUT_ROOT  = REPO_ROOT / "outputs_new"
CFG_DIR   = REPO_ROOT / "configs" / "locations_new"

# excluded scenes (by config name; edit as needed)
EXCLUDE = {"loc_01", "loc_03", "loc_05", "loc_05-2",
           "loc_07", "loc_07-2", "loc_09", "loc_20", "loc_22"}

FORCE_RERUN = True    # recompute every location this run; set back to False afterwards to resume-skip

args = SimpleNamespace(
    model="yolov8s.pt", imgsz=1280, conf=0.10, classes={1},
    nms_iou=0.70, assoc_gap=3,
    min_move_px=40.0,             # direction displacement gate: jitter <13px / real riders >50px; 40 sits in the empty band and blocks animal jitter (a 27px dog)
    cos_gate=0.5,
    max_time_gap_s=30.0,          # EXIF gap >30s breaks the chain (consecutive frame numbers are not consecutive in time)
    min_link_app_sim=0.30,        # appearance gate on every link (HSV fingerprint; color and brightness judged separately, min of the two — separates white vs dark clothing)
    direction_max_span_s=4.0,     # direction only from within-burst displacement (2s double-shot x2 margin); 6s chains were observed to merge across triggers, so tightened to 4
    trust_rescue_pair_s=0.0,      # experiment failed validation and was rolled back (queue artifact); keep 0
    second_pass=True,             # guided re-detection: riders whose 2s pair exists on disk but was missed by YOLO in the other frame;
                                  # re-detect that frame at low conf, accept only with spatial prior + HSV appearance check -> second_pass_dets.csv for audit
    second_pass_conf=0.05,        # second-pass confidence threshold
    stationary_filter=True,       # remove parked bicycles -> stationary_objects.csv for audit
    stationary_radius=30.0, stationary_hits=6,
    stationary_span_s=300.0, stationary_span_frames=50,
    stationary_min_density=0.6,   # a parked object appears in >=60% of its window's captures; busy chokepoints have low density and are kept
    reuse_detections=True,        # replay saved detections, skip YOLO; images re-read to refresh HSV fingerprints; second pass only touches a few hundred pair frames
    save_crops=True, save_viz=True, max_images=None,
)
todo = [c.stem for c in sorted(CFG_DIR.glob("loc_*.json"))
        if "old" not in c.stem.lower() and ".exclude" not in c.stem
        and c.stem not in EXCLUDE]
print(f"{len(todo)} locations to run:", ", ".join(todo))

In [ ]:
# Cell 3: Batch run (resumable; safe to leave unattended)
done, skipped, failed = [], [], []
for loc in todo:
    outdir = OUT_ROOT / loc
    if not FORCE_RERUN and (outdir / "scene_summary.json").exists():
        skipped.append(loc); continue
    img_dir = rrc.find_img_dir(DATA_ROOT, loc)
    if img_dir is None:
        print(f"[{loc}] image folder not found, skipping"); failed.append(loc); continue
    try:
        rrc.run_location(loc, img_dir, CFG_DIR / f"{loc}.json", outdir, args)
        done.append(loc)
    except Exception as e:
        print(f"[{loc}] failed: {e}"); failed.append(loc)
print(f"\nfinished {len(done)} | skipped (already done) {len(skipped)} | failed {failed or 'none'}")

In [ ]:
# Cell 4: All-location summary table
agg = rrc.aggregate_locations(OUT_ROOT, exclude=EXCLUDE)
agg.to_csv(OUT_ROOT / "all_locations_summary.csv", index=False)
display(rrc.all_locations_table(agg))
print("saved:", OUT_ROOT / "all_locations_summary.csv")

In [ ]:
# Cell 5: Comparison against the manual counts (reads data/manual_counts_new.csv)
import pandas as pd
mc = pd.read_csv(REPO_ROOT / "data" / "manual_counts_new.csv", dtype={"loc": str})
MANUAL = {}
for _, r in mc.dropna(subset=["fwd_total"]).iterrows():
    MANUAL[f"loc_{r['loc']}"] = (int(r.fwd_total), int(r.ww_total))
print(f"loaded manual benchmark for {len(MANUAL)} locations")
display(rrc.manual_comparison_table(agg, MANUAL))
print("red ΔWW cells = deviation above 8 percentage points; diagnose those locations first")

# facility-level comparison: pipeline dominant_space vs the manual facility split (same definition)
fct = rrc.facility_comparison_table(agg, mc)
if fct is not None:
    display(fct)
    print("BL=bike lane, SW=sidewalk, RD=roadway; manual columns = fwd+ww totals for that facility")
else:
    print("(dominant_space requires a re-run — see the FORCE_RERUN note in Cell 2)")

In [ ]:
# Cell 6: Summary chart — per-location WW rate (with CI) and facility split
import matplotlib.pyplot as plt
import numpy as np
rrc._mpl_style()
d = agg[agg.dir_known > 0].sort_values("ww_rate", ascending=True)
fig, ax = plt.subplots(figsize=(9, 0.4*len(d)+1.5))
y = np.arange(len(d))
err = np.array([ (d.ww_rate-d.ww_lo).tolist(), (d.ww_hi-d.ww_rate).tolist() ])
ax.barh(y, d.ww_rate, xerr=err, height=0.55, color="#2a78d6",
        error_kw=dict(ecolor="#52514e", lw=1.4, capsize=3))
for i,(_,r) in enumerate(d.iterrows()):
    ax.text(min(r.ww_hi+0.03,1.0), i, f"{r.against}/{r.dir_known}", va="center", fontsize=9, color="#52514e")
ax.set_yticks(y); ax.set_yticklabels(d.location)
ax.set_xlim(0,1); ax.set_xlabel("wrong-way rate (95% CI)")
ax.set_title("Wrong-way rate by location — displacement method (gated only)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7: Review-CSV metrics rollup (re-run after labeling each scene's review page)
import pandas as pd
rows = []
for d_ in sorted(OUT_ROOT.glob("loc_*")):
    res = rrc.analyze_review_csv(d_, d_.name)
    if res: rows.append(res)
if rows:
    rev = pd.DataFrame(rows)
    rev_display = rev.assign(**{
        "precision": rev.precision.map("{:.0%}".format),
        "manual_ww": rev.manual_ww.map("{:.1%}".format),
        "corrected_ww": rev.corrected_ww.map("{:.1%}".format)})
    display(rev_display)
    rev.to_csv(OUT_ROOT / "review_metrics.csv", index=False)
else:
    print("No labeled review_*.csv yet — generate a review page with Cell 6 of the single-location notebook, label it, and export")

In [ ]:
# Cell 8: Two-second-pair diagnostic — where exactly do direction-unknown riders come from
# The first run per location reads every photo's EXIF (a few minutes); afterwards it is cached in capture_index.csv
import pandas as pd
rows = []
for loc in todo:
    outdir = OUT_ROOT / loc
    if not (outdir / "riders.csv").exists():
        continue
    img_dir = rrc.find_img_dir(DATA_ROOT, loc)
    if img_dir is None:
        continue
    try:
        rows.append(rrc.pair_diagnostic(outdir, img_dir, loc))
    except Exception as e:
        print(f"[{loc}] diagnostic failed: {e}")
if rows:
    pdf = pd.DataFrame(rows)
    display(pdf)
    pdf.to_csv(OUT_ROOT / "pair_diagnostic.csv", index=False)
    tot = pdf[["unknown","no_pair_on_disk","pair_missed","too_little_motion","rescue_distrusted"]].sum()
    print("dataset totals:", dict(tot))
    print("Reading: pair_missed = the 2s pair exists but the detector missed the rider in one frame (detection recall: night/fast/occluded);")
    print("         no_pair_on_disk = no second capture within 4s exists on disk (camera did not fire / file missing);")
    print("         too_little_motion = both frames present but displacement below the gate (waiting/slow);")
    print("         rescue_distrusted = associated via burst rescue; direction deliberately not trusted")